# Customer Churn — Data Quality Audit & Exploratory Analysis

**Internship task:** Data Quality and Exploratory Analysis  
**Dataset:** supplied synthetic customer churn sample

## Decision question
Which customer groups might a fictional subscription team contact first to investigate avoidable churn?

This notebook profiles data quality, documents conservative cleaning decisions, runs validation checks, calculates descriptive statistics, and plots initial patterns. It is **not** a predictive model and does not establish causation.

**Source alignment note:** the internship guide uses names such as `customer_id`, `tenure_months`, `monthly_charge`, and `churned`; the supplied CSV uses `CustomerID`, `TenureMonths`, `MonthlyCharges`, and `Churn`. This notebook keeps the actual CSV column names. The sample has only 15 synthetic records, so findings are highly uncertain.

## 1. Load the source data

The original CSV is retained under `data/raw/`. Run from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "customer_churn_sample.csv").exists():
    ROOT = ROOT.parent

raw_path = ROOT / "data" / "raw" / "customer_churn_sample.csv"
clean_path = ROOT / "data" / "processed" / "cleaned_customer_churn.csv"

df = pd.read_csv(raw_path)
print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
display(df.head())

## 2. Profile columns, data types, missing values, and duplicates

In [ ]:
profile = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "distinct_non_null": df.nunique(dropna=True)
})
display(profile)
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate CustomerID values:", int(df["CustomerID"].duplicated().sum()))
display(df.describe(include="all").T)

## 3. Check validity rules and potential outliers

The IQR method flags unusual values for review; it does not prove that a value is erroneous. The domain checks are intentionally broad and their assumptions are documented.

In [ ]:
numeric_cols = ["Age", "TenureMonths", "MonthlyCharges", "TotalCharges", "SupportTickets"]

outlier_rows = []
for col in numeric_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((df[col] < lower) | (df[col] > upper)).sum())
    outlier_rows.append({
        "column": col, "Q1": q1, "Q3": q3,
        "IQR_lower": lower, "IQR_upper": upper,
        "flagged_count": count
    })
display(pd.DataFrame(outlier_rows).round(2))

checks = {
    "age outside broad 0–120 range": int(((df["Age"] < 0) | (df["Age"] > 120)).sum()),
    "negative tenure": int((df["TenureMonths"] < 0).sum()),
    "negative monthly charges": int((df["MonthlyCharges"] < 0).sum()),
    "negative total charges": int((df["TotalCharges"] < 0).sum()),
    "negative support tickets": int((df["SupportTickets"] < 0).sum()),
    "charge arithmetic mismatch beyond ₹0.01": int(
        (~np.isclose(
            df["TotalCharges"],
            df["MonthlyCharges"] * df["TenureMonths"],
            atol=0.011, rtol=0
        )).sum()
    )
}
display(pd.Series(checks, name="issue_count").to_frame())

## 4. Clean conservatively and validate

Cleaning assumptions:
1. Preserve the original source CSV unchanged.
2. Trim leading and trailing whitespace in text columns.
3. Standardize simple casing for `Gender`, `SubscriptionType`, and `Churn`.
4. Keep the supplied contract labels unchanged because the guide uses different category wording.
5. Do not impute values or remove records unless a documented quality issue requires it.
6. Validate key identifiers, allowed outcome labels, numeric ranges, and the charge arithmetic relationship.

In [ ]:
clean = df.copy()

for col in clean.select_dtypes(include=["object", "string"]).columns:
    clean[col] = clean[col].astype("string").str.strip()

clean["Gender"] = clean["Gender"].str.title()
clean["SubscriptionType"] = clean["SubscriptionType"].str.title()
clean["Churn"] = clean["Churn"].str.title()

# Explicit post-cleaning checks
assert clean["CustomerID"].notna().all(), "Missing customer ID detected"
assert clean["CustomerID"].is_unique, "Duplicate customer ID detected"
assert clean.isna().sum().sum() == 0, "Missing cells remain; investigate before imputation"
assert set(clean["Churn"].unique()).issubset({"Yes", "No"}), "Unexpected Churn label"
assert clean["Age"].between(0, 120).all(), "Age outside validation bounds"
assert (clean["TenureMonths"] >= 0).all(), "Negative tenure detected"
assert (clean["MonthlyCharges"] >= 0).all(), "Negative monthly charge detected"
assert (clean["TotalCharges"] >= 0).all(), "Negative total charge detected"
assert (clean["SupportTickets"] >= 0).all(), "Negative support ticket count detected"
assert np.all(np.abs(
    clean["TotalCharges"] - clean["MonthlyCharges"] * clean["TenureMonths"]
) <= 0.011), "Charge arithmetic mismatch detected"

clean_path.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(clean_path, index=False)
print("All validation assertions passed.")
print(f"Rows retained: {len(clean)} of {len(df)}")
print(f"Cleaned CSV written to: {clean_path}")

## 5. Descriptive statistics and churn patterns

In [ ]:
print("Numeric summary statistics")
display(clean[numeric_cols].describe().T.round(2))

churn_summary = clean.groupby("Churn").agg(
    customers=("CustomerID", "count"),
    average_tenure_months=("TenureMonths", "mean"),
    average_support_tickets=("SupportTickets", "mean"),
    average_monthly_charges=("MonthlyCharges", "mean")
).round(2)
display(churn_summary)

contract_summary = clean.groupby("ContractType").agg(
    customers=("CustomerID", "count"),
    churned=("Churn", lambda values: int((values == "Yes").sum()))
)
contract_summary["churn_rate_percent"] = (
    contract_summary["churned"] / contract_summary["customers"] * 100
).round(1)
display(contract_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

clean["Churn"].value_counts().reindex(["No", "Yes"], fill_value=0).plot(
    kind="bar", ax=axes[0], title="Customers by churn status"
)
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Customer count")

clean.boxplot(column="TenureMonths", by="Churn", ax=axes[1])
axes[1].set_title("Tenure by churn status")
axes[1].set_xlabel("Churn")
axes[1].set_ylabel("Tenure (months)")

(contract_summary["churn_rate_percent"]).plot(
    kind="bar", ax=axes[2], title="Churn rate by contract type"
)
axes[2].set_xlabel("Contract type")
axes[2].set_ylabel("Churn rate (%)")
fig.suptitle("")
plt.tight_layout()
plt.show()

## 6. Initial findings, recommendation, and limitations

- **Observed churn:** 7 of 15 customers (**46.7%**).
- **Average tenure:** 7.0 months for customers marked churned vs 29.1 months for those not marked churned.
- **Average support tickets:** 4.3 for churned vs 1.0 for non-churned customers.
- **Observed contract pattern:** Month-to-Month has 7 churned out of 7 records (100%); One Year and Two Year each have 0 churned in 4 records (0%).

### Cautious recommendation
For a small follow-up investigation, review recent-tenure, month-to-month customers with multiple support tickets to see whether proactive support is worth testing. Do not interpret these patterns as proof that the contract type or tickets cause churn, and do not generalize the rates from this sample.

### Limitations
- Only 15 synthetic records are available; a single customer materially changes subgroup rates.
- Synthetic records may encode simple patterns not representative of real customers.
- No outreach outcomes or longitudinal history are provided.
- This analysis is descriptive EDA, not churn prediction.

### Reproducibility
Run all notebook cells. The notebook reads the raw data, runs assertions, writes `data/processed/cleaned_customer_churn.csv`, and displays statistics and charts.